In [1]:
# Import SparkSession, which is the entry point for using PySpark
from pyspark.sql import SparkSession

# Create a SparkSession
# local[*] means Spark will use all available CPU cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IMDB Big Data Analysis") \
    .getOrCreate()

# Display Spark session details
spark


In [2]:
# Load the IMDB movies dataset from CSV file
# header=True tells Spark that the first row contains column names
# inferSchema=True automatically detects data types
df = spark.read.csv(
    "C:/Users/admin/OneDrive/Desktop/Study Material/BigData_Analysis_PySpark/Data/23000 IMDB Most Popular Movies.csv",
    header=True,
    inferSchema=True
)

# Display first few records to verify data loading
df.show(5)


+--------------------+----+--------+--------+--------------------+------+--------------------+-----------------+--------------------+-----+
|               movie|year|category|duration|              genres|rating|             tagline|         director|               stars|votes|
+--------------------+----+--------+--------+--------------------+------+--------------------+-----------------+--------------------+-----+
|            Range 15|2016|   TV-MA|      89|Action, Comedy, H...|   4.3|Veterans wake up ...|   Ross Patterson|Sean Astin, Keith...| 5010|
|Snake in the Eagl...|1978|      PG|      90|      Action, Comedy|   7.3|An orphan who has...|    Woo-Ping Yuen|Jackie Chan, Siu-...|12191|
|Twinkle Twinkle L...|1985|   TV-14|     105|      Action, Comedy|   6.2|5 HK cops (4 horn...|Sammo Kam-Bo Hung|Sammo Kam-Bo Hung...| 4164|
|       McHale's Navy|1997|      PG|     108|      Action, Comedy|   4.5|A retired Navy of...|     Bryan Spicer|Tom Arnold, Dean ...| 6891|
|            Fastlan

In [3]:
from pyspark.sql.functions import col, regexp_replace

# Convert year column to Integer
df = df.withColumn(
    "year",
    col("year").cast("int")
)

# Convert duration column to Integer (if it contains minutes)
df = df.withColumn(
    "duration",
    col("duration").cast("int")
)

# Convert rating column to Float
df = df.withColumn(
    "rating",
    col("rating").cast("float")
)

# Remove commas from votes (e.g., "1,234") and convert to Integer
df = df.withColumn(
    "votes",
    regexp_replace(col("votes"), ",", "").cast("int")
)



In [4]:
# Print the schema of the dataset
# This shows column names and their data types
df.printSchema()


root
 |-- movie: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- tagline: string (nullable = true)
 |-- director: string (nullable = true)
 |-- stars: string (nullable = true)
 |-- votes: integer (nullable = true)



In [5]:
# Count total number of movies in the dataset
# count() is an action that triggers Spark computation
total_movies = df.count()

print("Total number of movies:", total_movies)


Total number of movies: 23162


In [6]:
# Remove rows with missing values in important columns
# This helps avoid incorrect calculations during analysis
df = df.dropna(subset=["rating", "votes", "genres"])

# Verify cleaned data count
print("Records after cleaning:", df.count())


Records after cleaning: 22750


In [7]:
# Cache the dataset in memory
# This improves performance when the same data is used multiple times
df.cache()


DataFrame[movie: string, year: int, category: string, duration: int, genres: string, rating: float, tagline: string, director: string, stars: string, votes: int]

In [8]:
# Import count function for aggregation
from pyspark.sql.functions import count

# Group movies by genre and count number of movies in each genre
movies_by_genre = (
    df.groupBy("genres")
      .agg(count("*").alias("Total Movies"))
      .orderBy("Total Movies", ascending=False)
)

# Display top 10 genres by movie count
movies_by_genre.show(10)


+--------------------+------------+
|              genres|Total Movies|
+--------------------+------------+
|Comedy, Drama, Ro...|        1142|
|Animation, Action...|         859|
|     Comedy, Romance|         766|
|      Drama, Romance|         678|
|Crime, Drama, Mys...|         604|
|Animation, Advent...|         583|
|Action, Adventure...|         565|
|Action, Crime, Drama|         539|
|              Comedy|         499|
|Action, Comedy, C...|         457|
+--------------------+------------+
only showing top 10 rows



In [9]:
# Import avg function to calculate average values
from pyspark.sql.functions import avg

# Calculate average movie rating for each genre
avg_rating_by_genre = (
    df.groupBy("genres")
      .agg(avg("rating").alias("Average Rating"))
      .orderBy("Average Rating", ascending=False)
)

# Display genres with highest average ratings
avg_rating_by_genre.show(10)


+--------------------+-----------------+
|              genres|   Average Rating|
+--------------------+-----------------+
|Short, Action, Drama|9.399999618530273|
|       Short, Sci-Fi|              9.0|
| Documentary, Comedy|8.733333269755045|
|Animation, Short,...|8.699999809265137|
|Comedy, Sport, Ta...|8.699999809265137|
|Documentary, Biog...|8.600000381469727|
|Documentary, Real...|8.600000381469727|
|   Comedy, Talk-Show|8.550000190734863|
|Documentary, Myst...|              8.5|
|Animation, Family...|              8.5|
+--------------------+-----------------+
only showing top 10 rows



In [10]:
# Select relevant columns and sort movies by rating in descending order
top_rated_movies = (
    df.select("movie", "genres", "rating")
      .orderBy("rating", ascending=False)
)

# Display top 10 highest-rated movies
top_rated_movies.show(10)


+--------------------+--------------------+------+
|               movie|              genres|rating|
+--------------------+--------------------+------+
|        Breaking Bad|Crime, Drama, Thr...|   9.5|
|               Bluey|   Animation, Family|   9.5|
|The Last Drive-In...|Comedy, Fantasy, ...|   9.5|
|         Slumberkins|Animation, Short,...|   9.5|
|Sesame Street: Ge...|Animation, Short,...|   9.5|
|           Chernobyl|Drama, History, T...|   9.4|
|     Time Management|Short, Action, Drama|   9.4|
|    Band of Brothers| Drama, History, War|   9.4|
|            The Wire|Crime, Drama, Thr...|   9.3|
|          The Chosen|      Drama, History|   9.3|
+--------------------+--------------------+------+
only showing top 10 rows



In [11]:
# Identify most popular movies based on number of votes
top_voted_movies = (
    df.select("movie", "votes", "rating")
      .orderBy("votes", ascending=False)
)

# Display top 10 most voted movies
top_voted_movies.show(10)


+--------------------+-------+------+
|               movie|  votes|rating|
+--------------------+-------+------+
|The Shawshank Red...|2709155|   9.3|
|     The Dark Knight|2682192|   9.0|
|           Inception|2380160|   8.8|
|          Fight Club|2153179|   8.8|
|     Game of Thrones|2132768|   9.2|
|        Forrest Gump|2104931|   8.8|
|        Pulp Fiction|2079864|   8.9|
|        Breaking Bad|1932973|   9.5|
|          The Matrix|1932100|   8.7|
|The Lord of the R...|1894349|   8.8|
+--------------------+-------+------+
only showing top 10 rows



In [12]:
# Analyze number of movies released each year
movies_per_year = (
    df.groupBy("year")
      .count()
      .orderBy("year",ascending=False)
)

# Display first 10 years
movies_per_year.show(10)


+----+-----+
|year|count|
+----+-----+
|2023|  163|
|2022| 1129|
|2021|  913|
|2020|  814|
|2019|  951|
|2018|  945|
|2017|  868|
|2016|  826|
|2015|  761|
|2014|  725|
+----+-----+
only showing top 10 rows



In [13]:
from pyspark.sql.functions import avg

# Calculate average rating per year
avg_rating_year = (
    df.groupBy("year")
      .agg(avg("rating").alias("Average Rating"))
      .orderBy("year",ascending=False)
)

avg_rating_year.show(10)


+----+------------------+
|year|    Average Rating|
+----+------------------+
|2023| 6.352147236923499|
|2022| 6.200000003019828|
|2021| 6.254764517844703|
|2020|  6.25761671441193|
|2019| 6.390325974841474|
|2018|6.3574603209419855|
|2017|  6.31831797394335|
|2016| 6.393099278572397|
|2015| 6.335348232660281|
|2014| 6.339724143620195|
+----+------------------+
only showing top 10 rows



In [14]:
# Group movies into rating buckets
rating_distribution = (
    df.groupBy("rating")
      .count()
      .orderBy("rating",ascending=False)
)

rating_distribution.show(10)


+------+-----+
|rating|count|
+------+-----+
|   9.5|    5|
|   9.4|    3|
|   9.3|    6|
|   9.2|   10|
|   9.1|    9|
|   9.0|   27|
|   8.9|   27|
|   8.8|   44|
|   8.7|   85|
|   8.6|  111|
+------+-----+
only showing top 10 rows



In [15]:
# Filter movies with high rating and high votes
popular_high_rated = df.filter(
    (df.rating >= 8.0) & (df.votes >= 100000)
)

popular_high_rated.select(
    "movie", "rating", "votes"
).orderBy("rating", ascending=False).show(10)


+--------------------+------+-------+
|               movie|rating|  votes|
+--------------------+------+-------+
|        Breaking Bad|   9.5|1932973|
|    Band of Brothers|   9.4| 476804|
|           Chernobyl|   9.4| 777648|
|Avatar: The Last ...|   9.3| 323798|
|            The Wire|   9.3| 346365|
|The Shawshank Red...|   9.3|2709155|
|Scam 1992: The Ha...|   9.3| 146407|
|        The Sopranos|   9.2| 412118|
|       The Godfather|   9.2|1881334|
|     Game of Thrones|   9.2|2132768|
+--------------------+------+-------+
only showing top 10 rows



In [16]:
from pyspark.sql.functions import split, explode

# Split genres and explode into multiple rows
df_genres = df.withColumn(
    "genre",
    explode(split(df.genres, ","))
)

# Average rating per individual genre
genre_rating = (
    df_genres.groupBy("genre")
             .agg(avg("rating").alias("Average Rating"))
             .orderBy("Average Rating", ascending=False)
)

genre_rating.show(10)


+------------+------------------+
|       genre|    Average Rating|
+------------+------------------+
| Documentary| 8.300000190734863|
|   Talk-Show| 8.030769128065844|
|        News| 7.524999976158142|
|   Film-Noir|7.3625001311302185|
|   Game-Show| 7.361538492716276|
| Documentary| 7.317842341062933|
|       Short| 7.208888875113593|
|   Film-Noir| 7.201639355206098|
|       Adult| 7.099999904632568|
|   Biography| 7.077024805053206|
+------------+------------------+
only showing top 10 rows



In [17]:
# Analyze directors with at least 5 movies
director_rating = (
    df.groupBy("director")
      .agg(
          count("*").alias("Movies Count"),
          avg("rating").alias("Average Rating")
      )
      .filter("`Movies Count` >= 5")
      .orderBy("Average Rating", ascending=False)
)

director_rating.show(10)


+--------------------+------------+-----------------+
|            director|Movies Count|   Average Rating|
+--------------------+------------+-----------------+
|      Akira Kurosawa|          11|8.181818181818182|
|   Christopher Nolan|          11|8.154545437205922|
|         Chuck Jones|           5|8.100000095367431|
|   Quentin Tarantino|          11|8.072727246717973|
|     Rajkumar Hirani|           5|8.040000057220459|
|       James Cameron|          10|8.040000057220459|
|     Stanley Kubrick|          12|7.974999984105428|
|      Hayao Miyazaki|          11|7.972727255387739|
|     Charles Chaplin|          10|7.930000066757202|
|Krzysztof Kieslowski|           7|7.900000027247837|
+--------------------+------------+-----------------+
only showing top 10 rows



In [18]:
df = df.withColumn(
    "length Category",
    when(df.duration>=120,"Long Movie").otherwise("Short Movie")
)

length_rating = (
    df.groupBy("length_category")
    .agg(avg("rating").alias("Average Rating"))
)

NameError: name 'when' is not defined